# 01 - Data Audit
**Objective:** understand both datasets and verify data quality before any transformation.
Unpaid Tagging has no official definition: use the neutral terms *tagged* (1) / *non-tagged* (0).

Data files are confidential and must stay in `data/raw/` (git-ignored). Outputs of this notebook are cleared before committing.

In [ ]:
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 50)

PATH_A = "../data/raw/finanku_data_all.csv"
PATH_B = "../data/raw/finanku_data_validasi.csv"
TARGET, ID = "Unpaid Tagging", "Customer ID"

A = pd.read_csv(PATH_A)
B = pd.read_csv(PATH_B)
print("Data All:", A.shape, "| Data Validasi:", B.shape)
assert A.shape == B.shape == (7561, 22), "File shape changed; investigate before continuing"

## 1. Structure, types, samples

In [ ]:
for name, df in [("Data All", A), ("Data Validasi", B)]:
    print(f"\n=== {name} ===")
    print(df.dtypes.value_counts())
    display(df.drop(columns=ID).head(3))   # ID hidden on purpose
print("\nColumns only in All:", set(A.columns) - set(B.columns))
print("Columns only in Validasi:", set(B.columns) - set(A.columns))

## 2. Missing values, duplicates, ID integrity

In [ ]:
for name, df in [("All", A), ("Validasi", B)]:
    print(name, "| missing cells:", int(df.isna().sum().sum()),
          "| duplicate rows:", int(df.duplicated().sum()),
          "| duplicate IDs:", int(df[ID].duplicated().sum()))
print("Same ID set:", set(A[ID]) == set(B[ID]))
print("Same ID order:", bool((A[ID].values == B[ID].values).all()))

## 3. Shared columns match across files

In [ ]:
profile = ["Branch Code", "City", "Age", "Avg. Annual Income/Month"]
print("Profile identical:", all((A[c].values == B[c].values).all() for c in profile))
for q in ["Q2", "Q3", "Q4"]:
    cols = [c for c in A.columns if c.endswith(q)]
    print(q, "identical:", all((A[c].values == B[c].values).all() for c in cols))

## 4. Target distribution and label changes

In [ ]:
for name, df in [("All", A), ("Validasi", B)]:
    n1 = int(df[TARGET].sum())
    print(f"{name}: tagged={n1:,} ({n1/len(df):.2%}) non-tagged={len(df)-n1:,}")

print("\nCustomers with changed label:", int((A[TARGET].values != B[TARGET].values).sum()))
display(pd.crosstab(A[TARGET].rename("All"), B[TARGET].rename("Validasi"), margins=True))

## 5. Constant columns, value ranges, categories

In [ ]:
for name, df in [("All", A), ("Validasi", B)]:
    const = {c: df[c].unique().tolist() for c in df.columns if df[c].nunique() <= 1}
    print(name, "constant columns:", const)

display(A.drop(columns=ID).describe().T)
print(A["City"].value_counts())
print("Branch codes:", A["Branch Code"].nunique(),
      "| max cities per branch:", A.groupby("Branch Code")["City"].nunique().max())

## 6. Zero income and zero balances

In [ ]:
inc = "Avg. Annual Income/Month"
for name, df in [("All", A), ("Validasi", B)]:
    z = df[inc] == 0
    print(f"{name}: zero income={z.sum()} | tag rate zero-income={df.loc[z, TARGET].mean():.2%} "
          f"vs others={df.loc[~z, TARGET].mean():.2%}")

rows = []
for name, df, qs in [("All", A, ["Q1","Q2","Q3","Q4"]), ("Validasi", B, ["Q2","Q3","Q4","Q5"])]:
    for q in qs:
        rows.append({"dataset": name, "quarter": q,
                     "zero_balance_n": int((df[f"Balance {q}"] == 0).sum()),
                     "negative_balance_n": int((df[f"Balance {q}"] < 0).sum()),
                     "mean_balance": df[f"Balance {q}"].mean(),
                     "active_rate": df[f"ActiveMember {q}"].mean(),
                     "card_rate": df[f"HasCrCard {q}"].mean(),
                     "mean_products": df[f"NumOfProducts {q}"].mean()})
display(pd.DataFrame(rows).round(4))

## 7. Product counts, card flips, Q5 vs Q4

In [ ]:
def transitions(df, q0, q1, col):
    d = df[f"{col} {q1}"] - df[f"{col} {q0}"]
    return {"up": int((d > 0).sum()), "down": int((d < 0).sum()), "same": int((d == 0).sum())}

for q0, q1 in [("Q1","Q2"), ("Q2","Q3"), ("Q3","Q4")]:
    print(f"Products {q0}->{q1}:", transitions(A, q0, q1, "NumOfProducts"),
          "| Active:", transitions(A, q0, q1, "ActiveMember"))
print("Products Q4->Q5:", transitions(B, "Q4", "Q5", "NumOfProducts"),
      "| Active:", transitions(B, "Q4", "Q5", "ActiveMember"))

print("\nCard Q1==Q2==Q3 for all:",
      bool(((A["HasCrCard Q1"] == A["HasCrCard Q2"]) & (A["HasCrCard Q2"] == A["HasCrCard Q3"])).all()))
print("Customers without card Q1-Q3 (all have card in Q4):", int((A["HasCrCard Q3"] == 0).sum()))
print("Balance Q5 == Q4 for", int((B["Balance Q5"] == B["Balance Q4"]).sum()), "of", len(B), "customers")
print("NumOfProducts Q5 distribution:", B["NumOfProducts Q5"].value_counts().sort_index().to_dict())

## Findings and decisions
See `reports/data_quality_report.md`. Update it with your own interpretation after reviewing the outputs above.